<a href="https://colab.research.google.com/github/Song-yiJung/korean-ocr-lectures/blob/main/2026-08-pnu-workshop/session4/%ED%95%99%EB%B3%B4_%EC%82%AC%EB%8B%A4%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 학보 지면 사다리

부산대학교 여름워크숍 · 비정형 사료의 디지털화 · 4차시 16:18

---

## 세 지면을 차례로 넣습니다

| 단 | 자료 | 무엇이 다른가 |
|---|---|---|
| **1단** | 부산광복 60년 201쪽 | 가로쓰기 · 순한글 · 단 하나 |
| **2단** | 부산대학보 1957 **1면** | 세로쓰기 · 국한문 · 다단 |
| **3단** | 부산대학보 1957 **3면** | 위의 전부 + 졸업생 478명 명단 |

## 칸 차례

①② 준비 · 열쇠 → ③ 프롬프트 둘 → ④ 어제 코드를 도구로 → ⑤ 사다리 →
⑥ 프롬프트 비교 → ⑦ 명단면 → ⑧ 나란히 보기

---
# ① 준비

`!` 는 프로그램을 실행하라는 표시입니다. `pip` 이 도구를 깔고 `wget` 이 파일을 받습니다.

세 장을 받습니다. **1단은 어제 쓰던 그 쪽**입니다.

In [ ]:
!pip install -q google-cloud-vision google-genai pillow

저장소 = "https://raw.githubusercontent.com/Song-yiJung/korean-ocr-lectures/main"

!wget -q "{저장소}/2026-08-pnu-workshop/session4/data/hakbo1957_p1.jpg" -O "학보1면.jpg"
!wget -q "{저장소}/2026-08-pnu-workshop/session4/data/hakbo1957_p3.jpg" -O "학보3면.jpg"
!wget -q "{저장소}/2026-08-pnu-workshop/session4/data/busan60_p201.png" -O "부산광복_p201.jpg"

import os
for f in ("부산광복_p201.jpg", "학보1면.jpg", "학보3면.jpg"):
    크기 = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f"{f:20} {크기:>7,.0f}KB" + ("" if 크기 > 100 else "   ⚠ 받기 실패"))

print("\n준비 끝")

---
# ② 열쇠 등록

| | 열쇠 | 어디에 |
|---|---|---|
| 구글 비전 | `.json` **파일** | 드라이브 `keys` 폴더 |
| 제미나이 | 긴 **문자열** | 왼쪽 🔑 보안 비밀 |

오전 노트북 ②칸과 같습니다. **런타임이 새로 떠서 다시 불러올 뿐입니다.**

이 칸은 **세 도막**입니다.

| | |
|---|---|
| `drive.mount(…)` | 권한을 묻는 창이 뜹니다. 계정을 고르고 허용하십시오 |
| `glob` | `keys` 폴더에서 `.json` 으로 끝나는 것을 찾습니다. 파일 이름이 사람마다 달라서요 |
| `userdata.get(…)` | 🔑 에 넣어둔 문자열을 꺼냅니다. **이름이 한 글자라도 다르면 못 찾습니다** |

## 두 열쇠는 담기는 자리가 다릅니다

**비전은 환경변수**에 파일 위치만 적어둡니다. 비전 도구가 알아서 그 자리를 찾습니다.
**제미나이는 `GEMINI_KEY` 라는 이름표**에 문자열을 담아, 뒤 칸에서 직접 건네줍니다.

> 돌리고 나면 **두 줄**이 찍힙니다. 두 줄 다 「확인」이어야 끝까지 갑니다.

> 이 노트북은 ⑤칸부터 끝까지 두 열쇠가 모두 필요합니다.
> 「없음」이 하나라도 있으면 화면을 보시는 편이 낫습니다.

In [ ]:
import os, glob
from google.colab import userdata, drive

# ── 1. 드라이브 마운트 — 비전 열쇠가 여기 있습니다
drive.mount("/content/drive")

# ── 2. 구글 비전 — keys 폴더의 .json 파일
열쇠들 = glob.glob("/content/drive/MyDrive/keys/*.json")

if 열쇠들:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 열쇠들[0]
    비전상태 = f"확인   파일 {os.path.basename(열쇠들[0])}"
elif not os.path.isdir("/content/drive/MyDrive/keys"):
    비전상태 = "없음   keys 폴더가 안 보입니다 — 마운트한 계정이 다를 수 있습니다"
else:
    비전상태 = "없음   keys 폴더에 .json 이 없습니다"

# ── 3. 제미나이 — 왼쪽 🔑 보안 비밀의 문자열
try:
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_KEY = ""

제미나이상태 = f"확인   문자열 {len(GEMINI_KEY)}자" if GEMINI_KEY else "없음   등록명과 노트북 액세스 토글을 확인하십시오"

print(f"\n구글 비전   {비전상태}")
print(f"제미나이    {제미나이상태}")

if not 열쇠들 or not GEMINI_KEY:
    print("\n「없음」이 있으면 손을 들어 주십시오.")

---
# ③ 프롬프트 두 개를 나란히 둡니다

**이름표 둘**입니다. 아래 칸들이 이 이름으로 부릅니다.

| | |
|---|---|
| `현대어프롬프트` | 어제·오전에 쓰던 것. 다섯 줄짜리 규칙 |
| `신문프롬프트` | 세로쓰기·국한문·다단을 알려주는 것 |

`\"\"\"` 로 묶인 부분이 통째로 하나의 글입니다. **길이를 견줘 보십시오.**

바뀌는 것은 이 글뿐이고, 코드는 그대로입니다.

## `원본학번 = 478`

기계가 센 값이 아니라 **사람이 원본 지면을 보고 센 값**입니다.
⑦칸이 이것을 100%로 놓고 비전과 제미나이를 견줍니다.

기준선을 기계 쪽에 두면 **비전이 이미 흘린 것이 안 보입니다.**

In [ ]:
# ── 어제 쓰던 것 (현대어) ──────────────────────────────
현대어프롬프트 = """아래는 한국 현대 인쇄 자료를 OCR 로 읽은 결과다.
함께 첨부한 원본 이미지를 보면서 교정하라.
규칙:
1. 원문에 없는 내용을 절대 추가하지 마라.
2. 판독이 불확실한 글자는 추측하지 말고 □ 로 표시하라.
3. 문단 구분을 원문 그대로 유지하라.
4. 가운뎃점(·), 마침표, 숫자를 임의로 고치지 마라.
5. 설명·머리말·수정 목록을 붙이지 마라. 교정된 본문만 출력하라.

[OCR 결과]
"""

# ── 근대 신문용 (사료 지식이 들어간 것) ─────────────────
신문프롬프트 = """당신은 1950~60년대 한국 대학신문 판독에 능한 국어학자이자 아키비스트입니다.
이 자료는 '부산대학보 / 부대신문' 지면으로, 다음 특징을 가집니다.
  - 세로쓰기 (글자는 위→아래, 단은 오른쪽→왼쪽)
  - 국한문 혼용 (한글과 한자가 섞임, 구자체 포함)
  - 여러 단(段)으로 나뉜 신문 편집
  - 스캔 상태가 고르지 않음

첨부한 원본 이미지와 1차 Vision 추출 결과를 대조하여 아래 지침대로 교정·복원하십시오.

[판독 지침]
1. 판독 순서: 세로쓰기 원칙(각 단을 위→아래로, 지면 전체는 오른쪽→왼쪽).
   1차 결과의 줄 순서는 뒤엉켜 있을 수 있으니 참고만 하고,
   반드시 '이미지에 보이는 단 구성'에 따라 다시 배열하십시오.
2. 다단 구분: 서로 다른 기사의 문장을 섞지 마십시오.
3. 국한문 보존: 원문이 한자면 한자로, 한글이면 한글로 그대로 둡니다.
   구자체를 신자체로 바꾸지 마십시오. (學→学 ✕, 國→国 ✕)
4. 표·명단 보존: 졸업생 명단이나 통계표처럼 열이 반복되는 부분은
   한 줄로 뭉개지 말고 행과 열 구조를 유지하십시오.
5. 불확실 표시: 한 글자라도 확신할 수 없으면 그 글자 뒤에 [?] 를 붙이십시오.
   전혀 판독되지 않는 글자는 □ 로 두십시오.
6. 원문에 없는 내용을 절대 추가하지 마십시오.
7. 설명·머리말을 붙이지 말고 교정된 본문만 출력하십시오.

[Vision 1차 추출 결과]
"""

# ── 원본을 눈으로 센 값입니다. ⑦칸이 기준선으로 씁니다 ──────
원본학번 = 478                        # 3면 졸업생 명단의 인원

print(f"현대어 {len(현대어프롬프트):>5,}자")
print(f"신문   {len(신문프롬프트):>5,}자")

---
# ④ 어제 쓴 코드를 도구로 만듭니다

## `def` — 내가 만드는 도구

```
def 비전읽기(사진파일):
    ...
    return 글, 낱말수, 의심수
```

`def` 다음에 오는 것이 **도구의 이름**이고, 괄호 안이 **넣을 것**입니다.
`return` 이 돌려줄 것이고요. 한 번 만들어 두면 `비전읽기("학보1면.jpg")` 처럼 부릅니다.

어제는 이 내용이 ⑥⑧칸에 그냥 늘어서 있었습니다. **세 장을 돌려야 해서 도구로 묶었습니다.**
안에 든 것은 어제와 같습니다.

`\"\"\"…\"\"\"` 가 `def` 바로 아래 있으면 **그 도구가 하는 일을 적어둔 메모**입니다.

## `raise SystemExit(…)`

②칸에서 「없음」이 있었으면 **여기서 멈춥니다.**
이 노트북은 ⑤칸부터 끝까지 열쇠 둘이 모두 필요해서, 없으면 진행할 것이 없습니다.

## 고친 것은 한 줄

```
언어힌트 = ["ko", "zh-Hant", "ja"]     # 어제는 ["ko", "zh", "en"]
```

한자가 섞인 지면이라 **번체 한자와 일본어를 넣었습니다.**
자료가 어려워졌다고 코드를 다시 짜는 것이 아닙니다. 이 한 줄입니다.

In [ ]:
from google.cloud import vision
from google import genai
from google.genai import types

if not 열쇠들 or not GEMINI_KEY:
    raise SystemExit("②칸에서 「없음」이 있었습니다. 이 노트북은 열쇠 둘이 모두 필요합니다.")

비전 = vision.ImageAnnotatorClient()
제미나이 = genai.Client(api_key=GEMINI_KEY)

언어힌트 = ["ko", "zh-Hant", "ja"]     # 어제는 ["ko","zh","en"]
모델 = "gemini-3.5-flash"              # 404 가 나면 gemini-3.6-flash


def 비전읽기(사진파일):
    """모양만 보고 베낀다. 낱말별 자신감도 함께 받는다."""
    응답 = 비전.document_text_detection(
        image=vision.Image(content=open(사진파일, "rb").read()),
        image_context=vision.ImageContext(language_hints=언어힌트))
    낱말 = [w for 면 in 응답.full_text_annotation.pages
            for 덩 in 면.blocks for 문단 in 덩.paragraphs for w in 문단.words]
    의심 = [w for w in 낱말 if w.confidence < 0.80]
    return 응답.full_text_annotation.text, len(낱말), len(의심)


def 제미나이교정(사진파일, 지시, 비전결과):
    """원본 사진과 대조해 문맥으로 고친다."""
    응답 = 제미나이.models.generate_content(
        model=모델,
        contents=[types.Part.from_bytes(data=open(사진파일, "rb").read(),
                                        mime_type="image/jpeg"),
                  지시 + 비전결과],
        config=types.GenerateContentConfig(max_output_tokens=32768),
    )
    return 응답.text or ""


print("도구 둘 준비됨")

---
# ⑤ 사다리 — 세 지면을 차례로

비전만 돌립니다. 한 장에 1~2초입니다.

`사다리` 는 **이름과 파일을 짝지은 목록**입니다. 실습 노트북 ④칸의 `장들` 과 같은 꼴이고요.
`for 이름, 파일 in 사다리:` 로 짝을 풀어 받습니다.

| | |
|---|---|
| `결과[파일] = 글` | 파일 이름을 이름표 삼아 상자에 넣어둡니다. 뒤 칸이 꺼내 씁니다 |
| `"█" * int(비율 * 40)` | 글자에 곱하기를 하면 그만큼 되풀이됩니다 |
| `f"{비율:>9.0%}"` | 소수를 백분율로, 오른쪽 맞춤 |

**보실 것은 글자 수가 아니라 「자신 없어 한 낱말의 비율」입니다.**

In [ ]:
사다리 = [
    ("1단  부산광복 p201  (어제 그것)", "부산광복_p201.jpg"),
    ("2단  학보 1면       (세로·국한문·다단)", "학보1면.jpg"),
    ("3단  학보 3면       (+ 졸업생 478명 명단)", "학보3면.jpg"),
]

결과 = {}
print(f"{'':44}{'글자':>8}{'낱말':>8}{'자신없음':>10}")
print("-" * 72)
for 이름, 파일 in 사다리:
    글, 낱말수, 의심수 = 비전읽기(파일)
    결과[파일] = 글
    비율 = 의심수 / max(낱말수, 1)
    print(f"{이름:44}{len(글):>8,}{낱말수:>8,}{비율:>9.0%}  " + "█" * int(비율 * 40))

---
# ⑥ 같은 지면에 프롬프트 둘

**기사면(1면)** 에 두 프롬프트를 각각 넣습니다.
같은 사진, 같은 비전 결과, 같은 모델 — **바뀌는 것은 프롬프트뿐입니다.**

`.count("[?]")` 는 그 글이 몇 번 나오는지 셉니다.

> 한 번에 20~35초쯤 걸립니다. 두 번 도니 1분쯤 봐 주십시오.

In [ ]:
비전1면 = 결과["학보1면.jpg"]
교정1면 = {}

for 라벨, 지시 in (("현대어", 현대어프롬프트), ("신문용", 신문프롬프트)):
    답 = 제미나이교정("학보1면.jpg", 지시, 비전1면)
    교정1면[라벨] = 답
    print(f"{라벨:6} 프롬프트  →  {len(답):>6,}자   "
          f"[?] {답.count('[?]'):>3}개   □ {답.count('□'):>3}개")

print(f"\n(비전 원본은 {len(비전1면):,}자였습니다)")

---
# ⑦ 명단면

**3면**입니다. 오른쪽에 네 자리 학번이 늘어서 있습니다.
같은 두 프롬프트를 넣고, 학번이 몇 개나 남았는지 셉니다.

## `re` — 글 속에서 꼴을 찾기

```
네자리 = re.compile(r"\b[0-9]{4}\b")
```

| | |
|---|---|
| `[0-9]` | 숫자 한 글자 |
| `{4}` | 앞의 것이 **정확히 네 번** |
| `\b` | **낱말의 경계.** 양쪽에 붙여 「딱 네 자리」로 한정합니다 |

`\b` 가 없으면 여섯 자리 숫자 안에서도 네 자리를 잡아냅니다.

## 기준선은 원본 명단입니다

③칸의 `원본학번 = 478` 을 100%로 놓습니다. **비전 출력이 아닙니다.**

비전이 읽은 것을 100%로 잡으면 비전 단계에서 이미 흘린 것이 화면에서 사라집니다.
첫 줄과 둘째 줄의 간격이 **비전이 놓친 몫**이고, 그 아래가 **교정 단계에서 더 잃은 몫**입니다.

## `def 막대(수)`

`def` 는 도구를 하나 만드는 것입니다. 세 줄을 세 번 되풀이하는 대신
한 번 만들어 세 번 부릅니다. `"█" * int(비율 * 40)` — **글자에 곱하기를 하면 그만큼 되풀이됩니다.**

> **무슨 결과가 나오든 그대로 받으십시오.** 빈 응답이 나올 수도 있습니다.

In [ ]:
비전3면 = 결과["학보3면.jpg"]
교정3면 = {}

for 라벨, 지시 in (("현대어", 현대어프롬프트), ("신문용", 신문프롬프트)):
    교정3면[라벨] = 제미나이교정("학보3면.jpg", 지시, 비전3면)
    print(f"{라벨:6} 프롬프트  →  {len(교정3면[라벨]):>6,}자")

In [ ]:
import re
네자리 = re.compile(r"\b[0-9]{4}\b")

비전수 = len(네자리.findall(비전3면))

def 막대(수):
    비율 = 수 / 원본학번
    return f"{수:>4}개  ({비율:>5.1%})  " + "█" * int(비율 * 40)

print(f"원본 명단         {막대(원본학번)}")
print(f"비전 결과         {막대(비전수)}")
for 라벨, 답 in 교정3면.items():
    print(f"{라벨} 프롬프트   {막대(len(네자리.findall(답)))}")

print(f"\n원본은 졸업생 {원본학번}명입니다. 위 화면과 맞춰 보십시오.")

### 세는 규칙에 대해

이 세기는 **네 자리 숫자**만 셉니다. 학번이 다른 자릿수로 읽혔거나
숫자 사이에 공백이 끼어 들어간 것은 세지 못합니다.

또 본문 쪽의 연도(1957 따위)도 네 자리라 함께 잡힙니다.

**실제 손실은 화면 숫자보다 조금 더 큽니다.**

---
# ⑧ 나란히 보기

두 프롬프트가 같은 지면을 어떻게 읽었는지 앞부분만 견줍니다.

`교정3면.items()` 는 상자에서 **이름표와 값을 짝으로** 꺼냅니다.
`답[:260]` 은 앞에서 260자만 자르는 것이고요.

In [ ]:
for 라벨, 답 in 교정3면.items():
    print(f"─── {라벨} 프롬프트 ───")
    print(답[:260] if 답 else "(빈 응답)")
    print()